In [149]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

# ARIMA / SARIMA Modules
from statsmodels.tsa.stattools import adfuller
from pmdarima import auto_arima

# Prophet modules
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from sklearn.model_selection import ParameterGrid


# ML Models
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from pygam import LinearGAM, s
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

pio.renderers.default = "browser"


## Data Setup and Analysis

#### Load Data

In [4]:
def load_bitcoin_data(file_path, price_col):
    """
    Load Bitcoin data and automatically detect date column.

    Parameters:
        file_path (str): CSV or Excel file
        price_col (str): column name selected by user (e.g., 'Close')

    Returns:
        df: cleaned dataframe with ['ds', 'y']
    """

    # -----------------------------
    # 1. Load file
    # -----------------------------
    if file_path.endswith(".csv"):
        df = pd.read_csv(file_path)
    elif file_path.endswith((".xlsx", ".xls")):
        df = pd.read_excel(file_path)
    else:
        raise ValueError("Unsupported file format")

    original_columns = df.columns
    df.columns = [col.strip().lower() for col in df.columns]

    # -----------------------------
    # 2. Detect DATE column
    # -----------------------------
    date_keywords = ["date", "time", "timestamp", "datetime"]

    date_col = None

    # Step A: name-based detection
    for col in df.columns:
        if any(k in col for k in date_keywords):
            try:
                parsed = pd.to_datetime(df[col], errors="coerce")
                if parsed.notnull().mean() > 0.7:
                    date_col = col
                    df[col] = parsed
                    break
            except:
                continue

    # Step B: fallback data-based detection
    if date_col is None:
        scores = {}

        for col in df.columns:
            parsed = pd.to_datetime(df[col], errors="coerce")

            valid_ratio = parsed.notnull().mean()
            uniqueness = parsed.nunique() / len(parsed)

            score = valid_ratio * 0.7 + uniqueness * 0.3
            scores[col] = score

        date_col = max(scores, key=scores.get)
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

    # -----------------------------
    # 3. Validate user price column
    # -----------------------------
    price_col = price_col.lower()

    if price_col not in df.columns:
        raise ValueError(f"{price_col} not found. Available columns: {list(original_columns)}")

    # Convert to numeric safely
    df[price_col] = pd.to_numeric(df[price_col], errors="coerce")

    series = df[price_col]

    valid_ratio = series.notnull().mean()

    if valid_ratio < 0.7:
        raise ValueError(
            f"Selected column '{price_col}' is not valid numeric data "
            f"(only {valid_ratio:.2%} valid values)."
        )

    if series.nunique() < 5:
        raise ValueError(
            f"Selected column '{price_col}' does not look like a time-series signal "
            "(too few unique values)."
        )

    # -----------------------------
    # 🔥 NEW: PRICE-LIKE VALIDATION (FIX FOR VOLUME ISSUE)
    # -----------------------------

    skewness = series.skew()
    if abs(skewness) > 8:
        raise ValueError(
            f"Selected column '{price_col}' is too skewed ({skewness:.2f}) "
            "→ likely NOT a price column (possible volume or counts)"
        )

    ratio = series.max() / (series.median() + 1e-9)
    if ratio > 1e5:
        raise ValueError(
            f"Selected column '{price_col}' has abnormal range "
            "→ likely volume-like data, not price"
        )

    spike_ratio = (series.diff().abs() > series.std() * 5).mean()
    if spike_ratio > 0.2:
        raise ValueError(
            f"Selected column '{price_col}' is too noisy/spiky "
            "→ unlikely to be a valid price series"
        )

    # -----------------------------
    # 4. Clean dataframe
    # -----------------------------
    df = df[[date_col, price_col]].dropna()

    print(f"Detected date column: {date_col}")

    df = df.rename(columns={
        date_col: "ds",
        price_col: "y"
    })

    df = df.sort_values("ds")

    return df


In [5]:
file_path = "../btc_data/BTCUSD_1m_Binance.csv"

df = load_bitcoin_data(file_path, "Close")

df.head()

Detected date column: open time


,ds,y
0,2017-08-17 04:00:00,4261.48
1,2017-08-17 04:01:00,4261.48
2,2017-08-17 04:02:00,4280.56
3,2017-08-17 04:03:00,4261.48
4,2017-08-17 04:04:00,4261.48


#### Data Preprocessing

In [6]:
def preprocess_data(df):
    """
    Clean and prepare time series data
    """

    # Ensure datetime
    df["ds"] = pd.to_datetime(df["ds"], errors="coerce")

    # Drop invalid dates
    df = df.dropna(subset=["ds", "y"])

    # Sort
    df = df.sort_values("ds")

    # Remove duplicates (keep first)
    df = df.drop_duplicates(subset="ds")

    # Reset index
    df = df.reset_index(drop=True)

    return df

In [7]:
df = preprocess_data(df)
df.head()

,ds,y
0,2017-08-17 04:00:00,4261.48
1,2017-08-17 04:01:00,4261.48
2,2017-08-17 04:02:00,4280.56
3,2017-08-17 04:03:00,4261.48
4,2017-08-17 04:04:00,4261.48


In [8]:
def detect_frequency(df):
    """
    Detect dataset frequency automatically
    """
    diffs = df["ds"].diff().dropna()

    # most common time difference
    freq = freq = timedelta_to_freq(diffs.mode()[0])

    return freq

def timedelta_to_freq(td):
    seconds = int(td.total_seconds())

    if seconds == 60:
        return "1min"
    elif seconds == 300:
        return "5min"
    elif seconds == 900:
        return "15min"
    elif seconds == 3600:
        return "H"
    elif seconds == 86400:
        return "D"
    else:
        return f"{seconds}S"


In [9]:
print("Detected frequency:", detect_frequency(df))

Detected frequency: 1min


In [10]:
def resample_data(df, freq="D", target_col="y"):
    """
    Flexible resampling:
    - Supports single-column series (current use case)
    - Supports OHLC datasets if available in future
    """

    df = df.set_index("ds")

    # -----------------------------
    # CASE 1: OHLC exists
    # -----------------------------
    ohlc_cols = ["open", "high", "low", "close"]

    if all(col in df.columns for col in ohlc_cols):

        df = df.resample(freq).agg({
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last"
        })

        # if user selected one column, reduce to it
        if target_col.lower() in df.columns:
            df = df[[target_col.lower()]]
        else:
            raise ValueError(f"{target_col} not found in OHLC data")

    # -----------------------------
    # CASE 2: single column
    # -----------------------------
    else:
        df = df.resample(freq).mean()

        # clean missing values
        df[target_col] = df[target_col].ffill()

    df = df.reset_index()

    return df


In [11]:
freq = detect_frequency(df)
freq

'1min'

In [12]:
df = resample_data(df)

freq = detect_frequency(df)
print(df.head())
freq

          ds            y
0 2017-08-17  4358.630667
1 2017-08-18  4230.951715
2 2017-08-19  4070.712250
3 2017-08-20  4123.014063
4 2017-08-21  4035.014465


'D'

#### Visualization

In [13]:
def visualize_bitcoin_plotly(df, title="Bitcoin Price Over Time"):
    """
    Interactive Plotly visualization for time series data.
    """

    df = df.sort_values("ds")

    fig = go.Figure()

    # Main line chart
    fig.add_trace(
        go.Scatter(
            x=df["ds"],
            y=df["y"],
            mode="lines",
            name="Price",
            line=dict(width=2),
            hovertemplate=
                "<b>Time:</b> %{x}<br>" +
                "<b>Price:</b> %{y:.2f}<extra></extra>"
        )
    )

    # Layout styling (important for UX)
    fig.update_layout(
        title=title,
        xaxis_title="Time",
        yaxis_title="Price",
        hovermode="x unified", 
        template="plotly_dark",
        height=600
    )

    # Improve axis readability
    fig.update_xaxes(rangeslider_visible=True)

    fig.show()

In [13]:
visualize_bitcoin_plotly(df)

## Forecast Configurations

##  Refactor

In [14]:
from abc import ABC, abstractmethod

class BaseForecastModel(ABC):

    def __init__(self, df):
        self.df = df.copy()
        self.df = self.df.sort_values("ds")

    @abstractmethod
    def forecast(self, horizon, ci=0.95):
        pass

In [225]:
class MLModel(BaseForecastModel):

    def __init__(self, df, model_type="hybrid"):
        super().__init__(df)
        self.model_type = model_type

    # =====================================
    # SAFE FEATURES (NO STRINGS, NO DS IN XGB)
    # =====================================
    def _create_ml_features(self, df=None):

        df = self.df.copy() if df is None else df.copy()

        # =====================================
        # TIME FEATURES
        # =====================================
        df["dow"] = df["ds"].dt.dayofweek
        df["month"] = df["ds"].dt.month
        df["doy"] = df["ds"].dt.dayofyear
        df["is_weekend"] = df["dow"].isin([5, 6]).astype(int)

        # =====================================
        # CYCLICAL ENCODING
        # =====================================
        df["dow_sin"] = np.sin(2 * np.pi * df["dow"] / 7)
        df["dow_cos"] = np.cos(2 * np.pi * df["dow"] / 7)

        df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
        df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

        # =====================================
        # LAG FEATURES (CRITICAL FOR XGB)
        # =====================================
        for lag in [1, 3, 7]:
            df[f"lag_{lag}"] = df["y"].shift(lag)

        # =====================================
        # ROLLING FEATURES (SHIFTED TO AVOID LEAKAGE)
        # =====================================
        for w in [7, 14, 30]:
            df[f"roll_mean_{w}"] = df["y"].shift(1).rolling(w).mean()
            df[f"roll_std_{w}"] = df["y"].shift(1).rolling(w).std()

        # =====================================
        # CLEAN
        # =====================================
        return df.dropna()


    # =====================================
    # XGB MODEL
    # =====================================
    def _get_xgb(self):
        return XGBRegressor(
            n_estimators=800,
            max_depth=6,
            learning_rate=0.05,
        )

    # =====================================
    # HYBRID TRAINING (GAM + XGB)
    # =====================================
    def _train_hybrid(self, X_train, y_train):

        # -----------------------------
        # TREND MODEL (GAM)
        # -----------------------------
        t_train = np.arange(len(y_train)).reshape(-1, 1)

        gam = LinearGAM(s(0)).gridsearch(t_train, y_train)

        trend_train = pd.Series(gam.predict(t_train), index=y_train.index)

        # -----------------------------
        # RESIDUALS
        # -----------------------------
        residuals = y_train - trend_train

        # XGB training (IMPORTANT: ONLY numeric cols)
        xgb = self._get_xgb()
        xgb.fit(X_train, residuals)

        return gam, xgb

    # =====================================
    # FORECAST (NO RECURSION)
    # =====================================
    def forecast(self, horizon=30, ci=0.95, freq="D"):
        # =====================================
        # 1. SPLIT LIKE REAL FORECASTING
        # =====================================
        df = self._create_ml_features()

        split_idx = len(df)

        X = df.drop(columns=["y", "ds"])
        y = df["y"]

        # TRAIN ONLY (IMPORTANT FIX)
        X_train = X.copy()
        y_train = y.copy()

        # =====================================
        # 2. TRAIN HYBRID
        # =====================================
        gam, xgb = self._train_hybrid(X_train, y_train)

        # =====================================
        # 3. FUTURE TIME INDEX
        # =====================================
        future_dates = pd.date_range(
            self.df["ds"].iloc[-1],
            periods=horizon + 1,
            freq=freq
        )[1:]

        t_future = np.arange(len(y_train), len(y_train) + horizon).reshape(-1, 1)

        # =====================================
        # 4. GAM TREND
        # =====================================
        trend_future = gam.predict(t_future)

        # =====================================
        # 5. FUTURE FEATURES (SAFE)
        # =====================================
        future_df = pd.DataFrame({"ds": future_dates})

        full_df = pd.concat([self.df, future_df], ignore_index=True)
        full_df = self._create_ml_features(full_df)

        X_future = full_df.tail(horizon).drop(columns=["y", "ds"])

        # =====================================
        # 6. RESIDUAL PREDICTION
        # =====================================
        residual_future = xgb.predict(X_future)

        # =====================================
        # 7. FINAL OUTPUT
        # =====================================
        yhat = trend_future + residual_future

        std = np.std(y_train)

        return pd.DataFrame({
            "ds": future_dates,
            "yhat": yhat,
            "yhat_lower": yhat - 1.96 * std,
            "yhat_upper": yhat + 1.96 * std
        })

In [226]:
class NaiveModel(BaseForecastModel):

    def forecast(self, horizon, ci=0.95):
        last_value = self.df["y"].iloc[-1]

        future_dates = pd.date_range(
            start=self.df["ds"].iloc[-1],
            periods=horizon + 1,
            freq="D"
        )[1:]

        std = self.df["y"].std()
        z = 1.96 if ci == 0.95 else 1.64

        forecast = pd.DataFrame({
            "ds": future_dates,
            "yhat": [last_value] * horizon
        })

        forecast["yhat_lower"] = forecast["yhat"] - z * std
        forecast["yhat_upper"] = forecast["yhat"] + z * std

        return forecast

In [227]:
class ProphetModel(BaseForecastModel):

    def forecast(self, horizon, ci=0.95, freq="D"):

        # -----------------------------
        # Prepare data
        # -----------------------------
        df = self.df.copy()[["ds", "y"]].dropna()

        n = len(df)

        # -----------------------------
        # ADAPTIVE CHANGEPOINT FLEXIBILITY
        # -----------------------------
        if n < 200:
            cps = 0.5   # small data → more flexible
        elif n < 1000:
            cps = 0.1
        else:
            cps = 0.05  # large data → smoother

        # -----------------------------
        # ADAPTIVE SEASONALITY
        # -----------------------------
        is_hourly = freq and "H" in freq

        if is_hourly:
            daily_seasonality = True
            weekly_fourier = 10
        else:
            daily_seasonality = False
            weekly_fourier = 5

        # -----------------------------
        # VOLATILITY CHECK
        # -----------------------------
        volatility = df["y"].pct_change().std()

        seasonality_mode = "multiplicative"

        # if volatility > 0.05:
        #     seasonality_mode = "multiplicative"
        # else:
        #     seasonality_mode = "additive"

        print("seasonality mode", seasonality_mode)
        print("changepoint_prior_scale", cps)
        print("daily_seasonality", daily_seasonality)

        # -----------------------------
        # Initialize model
        # -----------------------------
        model = Prophet(
            seasonality_mode=seasonality_mode,
            changepoint_prior_scale=cps,
            interval_width=ci,
            daily_seasonality=daily_seasonality,
            weekly_seasonality=False,
            yearly_seasonality=False
        )


        # -----------------------------
        # ADD CUSTOM SEASONALITIES
        # -----------------------------

        # 1. Weekly crypto cycle (7 days)
        model.add_seasonality(
            name="weekly_crypto",
            period=7,
            fourier_order=5
        )

        # 2. Monthly liquidity cycle (~30 days)
        model.add_seasonality(
            name="monthly_cycle",
            period=30,
            fourier_order=8
        )

        # 3. Optional: 14-day momentum cycle
        model.add_seasonality(
            name="biweekly_cycle",
            period=14,
            fourier_order=5
        )

        # -----------------------------
        # Fit model
        # -----------------------------
        model.fit(df)

        # -----------------------------
        # Forecast future
        # -----------------------------
        future = model.make_future_dataframe(periods=horizon, freq=freq)
        forecast = model.predict(future)

        # -----------------------------
        # Extract only prediction horizon
        # -----------------------------
        forecast_df = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(horizon)

        return forecast_df

In [228]:
class SarimaModel(BaseForecastModel):

    # -----------------------------
    # 1. PREPARATION
    # -----------------------------
    def _prepare(self):
        df = self.df.copy().sort_values("ds").set_index("ds")

        # -------- Frequency --------
        freq = pd.infer_freq(df.index)

        if freq is None:
            median_diff = df.index.to_series().diff().median()

            if median_diff <= pd.Timedelta("1H"):
                freq = "H"
            elif median_diff <= pd.Timedelta("1D"):
                freq = "D"
            else:
                freq = "D"

        df = df.asfreq(freq)

        # -------- Missing --------
        df["y"] = df["y"].ffill()

        # -------- Valid values --------
        df = df[df["y"] > 0]

        # -------- Outliers --------
        df["y"] = self._remove_outliers(df["y"])

        return df, freq

    # -----------------------------
    # 2. STATIONARITY CHECK
    # -----------------------------
    def _check_stationarity(self, series):
        result = adfuller(series.dropna())
        p_value = result[1]

        return p_value < 0.05, p_value

    # -----------------------------
    # 3. MAKE STATIONARY (AUTO d)
    # -----------------------------
    def _make_stationary(self, series, max_diff=2):

        diff_series = series.copy()
        d = 0

        for _ in range(max_diff):
            is_stationary, _ = self._check_stationarity(diff_series)

            if is_stationary:
                break

            diff_series = diff_series.diff().dropna()
            d += 1

        return diff_series, d

    # -----------------------------
    # 4. SEASONALITY
    # -----------------------------
    def _infer_seasonality(self, freq):

        if freq in ["H"]:
            return 24
        elif freq in ["D"]:
            return 7
        elif "min" in str(freq):
            return 60
        else:
            return 1

    # -----------------------------
    # 5. OUTLIERS
    # -----------------------------
    def _remove_outliers(self, series):

        q1 = series.quantile(0.01)
        q99 = series.quantile(0.99)

        return series.clip(q1, q99)

    # -----------------------------
    # 6. FORECAST
    # -----------------------------
    def forecast(self, horizon, ci=0.95, freq="D"):

        df, freq = self._prepare()
        series = df["y"]

        # -------- Log transform --------
        log_series = np.log(series)

        # -------- Stationarity --------
        diff_series, d = self._make_stationary(log_series)

        # -------- Seasonality --------
        m = self._infer_seasonality(freq)

        # -------- Model --------
        model = auto_arima(
            diff_series,
            d=d,
            seasonal=True,
            m=m,
            trace=False,
            error_action="ignore",
            suppress_warnings=True,
            stepwise=True,
            max_p=3,
            max_q=3
        )

        # -------- Forecast --------
        forecast, conf_int = model.predict(
            n_periods=horizon,
            return_conf_int=True,
            alpha=1 - ci
        )

        # -------- Inverse transform --------
        last_log = log_series.iloc[-1]
        forecast_log = last_log + np.cumsum(forecast)

        yhat = np.exp(forecast_log)
        
        lower_log = forecast_log + conf_int[:, 0]
        upper_log = forecast_log + conf_int[:, 1]

        lower = np.exp(lower_log)
        upper = np.exp(upper_log)


        # -------- Dates --------
        future_dates = pd.date_range(
            start=df.index[-1],
            periods=horizon + 1,
            freq=freq
        )[1:]

        return pd.DataFrame({
            "ds": future_dates,
            "yhat": yhat,
            "yhat_lower": lower,
            "yhat_upper": upper
        })


In [229]:
class ForecastEngine:

    def __init__(self, df):
        self.df = df.copy().sort_values("ds")

        # -----------------------------
        # MODEL REGISTRY
        # -----------------------------
        self.models = {
            "naive": NaiveModel,
            "sarima": SarimaModel,
            "prophet": ProphetModel,
            "xgb": lambda df: MLModel(df, "xgb"),
            "hybrid": lambda df: MLModel(df, "hybrid"),
        }

    # =====================================
    # 1. TECHNICAL INDICATORS (SHARED)
    # =====================================
    def add_indicators(self, use_sma=False, use_ema=False, window=20):

        if use_sma:
            self.df["SMA"] = self.df["y"].rolling(window).mean()

        if use_ema:
            self.df["EMA"] = self.df["y"].ewm(span=window, adjust=False).mean()

    # =====================================
    # 2. GET MODEL
    # =====================================
    def get_model(self, model_type, df=None):

        df = df if df is not None else self.df

        if model_type not in self.models:
            raise ValueError(f"Unsupported model: {model_type}")

        model_class = self.models[model_type]

        return model_class(df)
    

    def _get_freq(self):
        freq = pd.infer_freq(self.df["ds"])
        return freq if freq is not None else "D"


    # =====================================
    # 3. FORECAST (MAIN ENTRY)
    # =====================================
    def forecast(self, model_type="naive", horizon=30, ci=0.95, df=None):

        model = self.get_model(model_type, df=df)

        freq = self._get_freq()


        print(f"Using model: {model_type}")
        print(f"Horizon: {horizon} steps")

        forecast_df = model.forecast(horizon=horizon, ci=ci, freq="D")

        return forecast_df

    # =====================================
    # 4. EVALUATION (UNIVERSAL)
    # =====================================
    def evaluate(self, y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)

        min_len = min(len(y_true), len(y_pred))

        y_true = y_true[:min_len]
        y_pred = y_pred[:min_len]

        mae = np.mean(np.abs(y_true - y_pred))
        rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

        # safer MAPE
        mape = mean_absolute_percentage_error(y_true, y_pred) * 100
        # mape = np.mean(
        #     np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))
        # ) * 100

        r2 = r2_score(y_true, y_pred)

        return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "R2 Score": r2}


    # =====================================
    # 5. VISUALIZATION (SHARED)
    # =====================================
    def plot_backtest(self, train, test, test_forecast, future_forecast=None):
        fig = go.Figure()

        # =====================================
        # TRAIN
        # =====================================
        fig.add_trace(go.Scatter(
            x=train["ds"],
            y=train["y"],
            mode="lines",
            name="Train",
            line=dict(color="blue")
        ))

        # =====================================
        # TEST (ACTUAL)
        # =====================================
        fig.add_trace(go.Scatter(
            x=test["ds"],
            y=test["y"],
            mode="lines",
            name="Test (Actual)",
            line=dict(color="orange")
        ))

        # =====================================
        # TEST PREDICTION
        # =====================================
        fig.add_trace(go.Scatter(
            x=test_forecast["ds"],
            y=test_forecast["yhat"],
            mode="lines",
            name="Test Prediction",
            line=dict(dash="dash", color="red")
        ))

        # =====================================
        # FUTURE FORECAST (OPTIONAL)
        # =====================================
        if future_forecast is not None:
            fig.add_trace(go.Scatter(
                x=future_forecast["ds"],
                y=future_forecast["yhat"],
                mode="lines",
                name="Future Forecast",
                line=dict(dash="dot", color="green")
            ))

        # =====================================
        # SPLIT LINES (IMPORTANT VISUAL CUE)
        # =====================================
        train_end = train["ds"].iloc[-1]
        test_end = test["ds"].iloc[-1]

        fig.add_shape(
            type="line",
            x0=train_end,
            x1=train_end,
            y0=0,
            y1=1,
            xref="x",
            yref="paper",
            line=dict(color="white", width=2, dash="dash")
        )


        fig.add_shape(
            type="line",
            x0=test_end,
            x1=test_end,
            y0=0,
            y1=1,
            xref="x",
            yref="paper",
            line=dict(color="red", width=2, dash="dash")
        )


        # =====================================
        # LAYOUT
        # =====================================
        fig.update_layout(
            title="Train / Test / Forecast Split Visualization",
            hovermode="x unified",
            template="plotly_dark",
            height=650
        )

        fig.update_xaxes(rangeslider_visible=True)

        fig.show()


    # =====================================
    # 6. BACKTEST (NEW 🔥)
    # =====================================
    def backtest(self, model_type="naive", test_size=30, ci=0.95):
        # -----------------------------
        # SPLIT
        # -----------------------------
        train = self.df.iloc[:-test_size].copy()
        test = self.df.iloc[-test_size:].copy()

        # -----------------------------
        # TRAIN MODEL
        # -----------------------------
        model = self.get_model(model_type, df=train)

        test_forecast = model.forecast(horizon=test_size, ci=ci)

        # -----------------------------
        # ALIGN TEST PREDICTIONS
        # -----------------------------
        test_merged = test.merge(test_forecast, on="ds", how="inner")

        metrics = self.evaluate(test_merged["y"], test_merged["yhat"])

        print("Backtest Results:")
        print(metrics)

        return train, test, test_forecast, metrics


    # =====================================
    # 7. AVAILABLE MODELS
    # =====================================
    def available_models(self):
        return list(self.models.keys())
    

    def model_info(self):
        return {
            "naive": "Baseline model (fast, simple)",
            "sarima": "Statistical model (good for seasonality)",
            "prophet": "Trend + seasonality (robust)",
            "xgb": "Machine learning model",
            "GAM": "Kernel-based ML model",
            "hybrid": "Stacked ML (best accuracy, slower)"
        }

In [230]:
df_sample = df.copy()
len(df_sample)

2978

In [231]:
engine = ForecastEngine(df_sample)
future_model = engine.get_model("hybrid", df=engine.df)
future_forecast = future_model.forecast(horizon=90)

  0% (0 of 11) |                         | Elapsed Time: 0:00:00 ETA:  --:--:--
 18% (2 of 11) |####                     | Elapsed Time: 0:00:00 ETA:   0:00:00
 45% (5 of 11) |###########              | Elapsed Time: 0:00:00 ETA:   0:00:00
 72% (8 of 11) |##################       | Elapsed Time: 0:00:00 ETA:   0:00:00
100% (11 of 11) |########################| Elapsed Time: 0:00:00 Time:  0:00:00


In [232]:
train, test, test_forecast, metrics = engine.backtest(
    model_type="hybrid",
    test_size=590
)

  0% (0 of 11) |                         | Elapsed Time: 0:00:00 ETA:  --:--:--
 27% (3 of 11) |######                   | Elapsed Time: 0:00:00 ETA:   0:00:00
 54% (6 of 11) |#############            | Elapsed Time: 0:00:00 ETA:   0:00:00
 81% (9 of 11) |####################     | Elapsed Time: 0:00:00 ETA:   0:00:00
100% (11 of 11) |########################| Elapsed Time: 0:00:00 Time:  0:00:00


Backtest Results:
{'MAE': np.float64(24381.992886797136), 'RMSE': np.float64(28634.371117924406), 'MAPE': 25.82622438919663, 'R2 Score': -0.9310271959301755}


In [188]:
train, test, test_forecast, metrics = engine.backtest(
    model_type="hybrid",
    test_size=590
)

  0% (0 of 11) |                         | Elapsed Time: 0:00:00 ETA:  --:--:--
 18% (2 of 11) |####                     | Elapsed Time: 0:00:00 ETA:   0:00:00
 36% (4 of 11) |#########                | Elapsed Time: 0:00:00 ETA:   0:00:00
 63% (7 of 11) |###############          | Elapsed Time: 0:00:00 ETA:   0:00:00
 90% (10 of 11) |#####################   | Elapsed Time: 0:00:00 ETA:   0:00:00
100% (11 of 11) |########################| Elapsed Time: 0:00:00 Time:  0:00:00


Backtest Results:
{'MAE': np.float64(24381.27798712553), 'RMSE': np.float64(28633.056434476202), 'MAPE': 25.82596707686452, 'R2 Score': -0.9308498823650952}


In [233]:
engine.plot_backtest(
    train,
    test,
    test_forecast,
    future_forecast
)

In [27]:


# # Add indicators
# engine.add_indicators(use_sma=True, use_ema=True)

# # Forecast
# forecast = engine.forecast("hybrid", horizon=90)

# # Plot

# # Backtest
# forecast_bt, metrics = engine.backtest("sarima", test_size=90)

# engine.plot(forecast, use_sma=True, use_ema=True)

Using model: sarima
Horizon: 90 steps


KeyboardInterrupt: 